# Etapa 2, E4 (osoba 2): LLM vyberie hazard a produkt z 10 kandidátov E12kens

Beží na datalabe (Quadro RTX 4000, 8 GB). Model **Qwen2.5-3B-Instruct** vo float16 (asi 6 GB na disku aj v GPU).
Všetko je na **valid**; test sa tu nespúšťa (spustí ho osoba 1 raz na konci).

**Postup:** pri štarte servera zvoľ konfiguráciu s GPU, otvor tento notebook v priečinku `notebooks/` a spúšťaj bunky postupne zhora (Shift+Enter). Pred spustením si prečítaj poznámku nad každou bunkou.

| Bunka | Čo robí | Čas |
| --- | --- | --- |
| 1 | nastavenie priečinka modelov, kontrola nahraných súborov | hneď |
| 2 | skúška celej cesty bez GPU a bez LLM | pár sekúnd |
| 3 | inštalácia torch a transformers (len ak chýbajú) | 3 až 5 min |
| 4 | kontrola GPU | pár sekúnd |
| 5 | stiahnutie modelu do tvojho priečinka (len prvýkrát) | asi 5 min |
| 6 | skúška s LLM na 50 správach | asi 1 min |
| 7 | celý valid (565 správ × 2 štítky) | asi 5 až 10 min |
| 8 | prahy istoty z uložených odpovedí a vyhodnotenie | asi 1 min |
| 9 | zbalenie výsledkov na stiahnutie | hneď |
| 10 | (na konci projektu) zmazanie modelu | hneď |

Každá odpoveď LLM sa hneď ukladá do `results/etapa2/llm_cache/`. Keď sa relácia preruší, spusti bunky 1, 4 a tú, ktorá bežala: pokračuje sa tam, kde sa skončilo.

**Bunka 1:** ak chceš model inde, zmeň `PRIECINOK_MODELOV`. Všetko, čo sa stiahne (model aj pomocné súbory Hugging Face), pôjde len do tohto priečinka. Bunka vypíše, či sú nahrané všetky potrebné súbory (všetky musia byť `OK`).

In [ ]:
import glob, importlib.util, os, re, shutil, sys, time, zipfile

# >>> nastavenie <<<
PRIECINOK_MODELOV = "/home/jovyan/work/SpracovaniePrirodzenehoJazyka/zadanie_2etapa/llm_modely"
MODEL = "Qwen/Qwen2.5-3B-Instruct"

MODEL_CESTA = os.path.join(PRIECINOK_MODELOV, MODEL.split("/")[-1])
os.makedirs(PRIECINOK_MODELOV, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(PRIECINOK_MODELOV, "hf_home")  # nie do ~/work/.cache
os.environ["PIP_NO_CACHE_DIR"] = "1"  # pip si inštalačné súbory neodkladá bokom
KRATKY = re.sub(r"[^\w]", "", MODEL.split("/")[-1])[:20]  # časť názvu systému, napr. Qwen253BInstruct

# pracujeme z koreňa projektu (priečinok, v ktorom sú src/, data/, results/)
KOREN = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(KOREN)
print("koreň projektu:", KOREN)
print("priečinok modelov:", PRIECINOK_MODELOV)
potrebne = ["src/spolocne.py", "src/e3_hierarchia.py", "src/e4_llm.py", "src/vyhodnot.py",
            "data/incidents_valid.csv", "results/etapa2/kandidati/E12kens_valid.csv",
            "results/etapa2/predikcie/E12kens_valid.csv", "requirements_e4.txt"]
for s in potrebne:
    print("OK    " if os.path.exists(s) else "CHÝBA ", s)
assert all(os.path.exists(s) for s in potrebne), "niektorý súbor chýba, skontroluj štruktúru priečinkov"

**Bunka 2:** skúška bez LLM (`--backend skuska` vždy vyberie kandidáta č. 1). Overí, že sa dáta a kandidáti správne načítajú. Očakávaný výsledok: riadky `E12kens` a `E4` majú rovnaké skóre a „zmenené 0“.

In [ ]:
!{sys.executable} src/e4_llm.py --backend skuska --limit 50

**Bunka 3:** inštalácia `torch` a `transformers` (verzie v `requirements_e4.txt`), len ak ešte nie sú nainštalované. Prvýkrát to trvá 3 až 5 minút (sťahuje sa asi 3 GB). Ak balíčky po reštarte servera zmiznú, bunka ich nainštaluje znova.

In [ ]:
chybaju = [b for b in ("torch", "transformers") if importlib.util.find_spec(b) is None]
if chybaju:
    print("inštalujem:", chybaju)
    !{sys.executable} -m pip install --no-cache-dir -q -r requirements_e4.txt
else:
    print("torch a transformers už sú nainštalované")

**Bunka 4:** kontrola, či torch vidí GPU a vie na nej počítať. Musí vypísať `výpočet na GPU: OK`. Ak skončí chybou, pozri bunku 4b.

In [ ]:
import torch, transformers

print("torch", torch.__version__, "| CUDA v torch", torch.version.cuda, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "torch nevidí GPU: je zvolená konfigurácia s GPU? Inak bunka 4b."
print("GPU:", torch.cuda.get_device_name(0))
# násobenie matíc vo float16 na GPU a porovnanie s tým istým výpočtom na CPU
a = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
c = a @ a
assert torch.isfinite(c).all(), "výpočet vrátil NaN/inf"
ref = a.float().cpu() @ a.float().cpu()
odchylka = (c.float().cpu() - ref).abs().max().item()
print(f"max. odchýlka od CPU: {odchylka:.3f} (hodnoty sú v priemere okolo {ref.abs().mean():.0f})")
assert odchylka < 1.0, "výsledok na GPU sa líši od CPU"
del a, c
torch.cuda.empty_cache()
print("výpočet na GPU: OK")

**Bunka 4b (len ak bunka 4 zlyhala):** nainštalovaná verzia torch je pre CUDA 13 a s ovládačom datalabu nemusí fungovať. Odkomentuj riadok (zmaž `# `), spusti ho, potom **Kernel → Restart Kernel** a spusti znova bunky 1 a 4. Do správy zapíšeme, ktorá verzia torch sa nakoniec použila.

In [ ]:
# !{sys.executable} -m pip install --no-cache-dir --force-reinstall torch --index-url https://download.pytorch.org/whl/cu126

**Bunka 5:** stiahne model do `PRIECINOK_MODELOV` (asi 6 GB). Ak už je stiahnutý, len to rýchlo overí a nič nesťahuje.

In [ ]:
from huggingface_hub import snapshot_download

t = time.time()
snapshot_download(MODEL, local_dir=MODEL_CESTA,
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*", "LICENSE*"])
velkost = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(PRIECINOK_MODELOV) for f in fs)
print(f"model je v {MODEL_CESTA}")
print(f"priečinok modelov zaberá {velkost / 1024**3:.1f} GB, trvalo {time.time() - t:.0f} s")

**Bunka 6:** skúška s LLM na prvých 50 správach. Overí, že model beží, a ukáže rýchlosť. Skóre na 50 správach je len orientačné. Dôležité je, aby `p_mimo_pismen` v detaile nebolo vysoké a aby beh nespadol. Odpovede sa uložia a celý beh v bunke 7 ich znova použije.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --limit 50

**Bunka 7:** celý valid (565 správ × 2 štítky = 1 130 otázok). LLM vždy použije svoj výber (prah 0). Výsledok sa uloží ako systém `E4_Qwen253BInstruct_E12kens`.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA}

**Bunka 8:** varianty s prahom istoty: LLM zmení odpoveď základného systému, len keď si je istý aspoň na 50, 70 alebo 90 %. Počíta sa z uložených odpovedí, GPU netreba. Potom vyhodnotenie všetkých variantov oproti E12kens s bootstrapom (1 000 výberov). Prah sa vyberá na valid, preto bude výsledok mierne optimistický; zapíšeme to do zápisníka.

In [ ]:
for prah in [0.5, 0.7, 0.9]:
    print(f"\n===== prah {prah} =====")
    !{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --prah {prah}

systemy = " ".join([f"E4_{KRATKY}_E12kens"] + [f"E4_{KRATKY}_E12kens_p{p:03d}" for p in (50, 70, 90)])
print("\n===== vyhodnotenie na valid =====")
!{sys.executable} src/vyhodnot.py --split valid E12kens {systemy} --porovnaj E12kens --bootstrap 1000 --uloz results/etapa2/llm/vyhodnotenie_valid.csv

**Bunka 9:** zbalí výsledky do `vysledky_e4.zip` v koreni projektu. Ten súbor stiahni (pravý klik → Download) a prines späť. Výpis tejto bunky aj bunky 8 môžeš skopírovať aj priamo.

In [ ]:
subory = sorted(glob.glob("results/etapa2/llm_cache/*") + glob.glob("results/etapa2/llm/*")
                + glob.glob("results/etapa2/predikcie/E4_*") + glob.glob("results/etapa2/logy/E4_*"))
with zipfile.ZipFile("vysledky_e4.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for s in subory:
        z.write(s)
        print(s)
print(f"\n{len(subory)} súborov v {os.path.join(KOREN, 'vysledky_e4.zip')}")

**Bunka 10 (až na konci projektu):** zmaže celý priečinok s modelom. Odkomentuj riadky a spusti. Nainštalované balíčky sa dajú odstrániť príkazom `pip uninstall torch transformers`.

In [ ]:
# shutil.rmtree(PRIECINOK_MODELOV)
# print("zmazané:", PRIECINOK_MODELOV)